# AI Route Optimization - Exploratory Data Analysis
Explores the synthetic delivery dataset used to train route prediction models.
Models: Random Forest, XGBoost, LSTM, K-Means Clustering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/synthetic_route_dataset.csv')
print('Shape:', df.shape)
df.head()

## 1. Dataset Overview
Verify minimum dataset requirements: 1000+ records, 10+ drivers, 50+ locations.

In [ ]:
print('Total records:', len(df))
print('Unique drivers:', df['driver_id'].nunique())
print('Unique locations:', df['location_id'].nunique())
print('Date range:', df['date'].min(), 'to', df['date'].max())
print('Columns:', list(df.columns))
df.describe()

## 2. Traffic Distribution and Impact on Travel Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors_map = ['#22c55e', '#f59e0b', '#ef4444']
df['traffic_category'].value_counts().plot(kind='bar', ax=axes[0], color=colors_map)
axes[0].set_title('Traffic Category Distribution')
axes[0].set_xlabel('Traffic Level')
axes[0].set_ylabel('Number of Trips')
df.groupby('traffic_category')['historical_travel_time_minutes'].mean().plot(kind='bar', ax=axes[1], color=colors_map)
axes[1].set_title('Average Travel Time by Traffic Level')
axes[1].set_xlabel('Traffic Level')
axes[1].set_ylabel('Minutes')
plt.tight_layout()
plt.savefig('../data/traffic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Driver Performance Analysis (Driver Features)

In [ ]:
driver_stats = df.groupby('driver_id').agg(
    avg_speed=('average_speed_kmph', 'mean'),
    avg_efficiency=('route_efficiency_score', 'mean'),
    total_trips=('driver_id', 'count'),
    avg_distance=('distance_km', 'mean')
).reset_index().sort_values('avg_efficiency', ascending=False)

print('Driver Performance Summary:')
print(driver_stats.to_string(index=False))

plt.figure(figsize=(12, 4))
plt.bar(driver_stats['driver_id'], driver_stats['avg_efficiency'], color='#3b82f6')
plt.title('Average Route Efficiency Score by Driver')
plt.xlabel('Driver ID')
plt.ylabel('Efficiency Score (0-1)')
plt.tight_layout()
plt.show()

## 4. Time Features - Delivery Patterns by Day of Week

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['day_of_week'].value_counts().reindex(day_order)
plt.figure(figsize=(10, 4))
colors = ['#3b82f6' if d not in ('Saturday', 'Sunday') else '#94a3b8' for d in day_order]
day_counts.plot(kind='bar', color=colors)
plt.title('Delivery Trips by Day of Week (Weekdays vs Weekends)')
plt.xlabel('Day of Week')
plt.ylabel('Number of Trips')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Route Features - Distance vs Travel Time

In [ ]:
traffic_colors = {'Low': '#22c55e', 'Medium': '#f59e0b', 'High': '#ef4444'}
plt.figure(figsize=(10, 5))
for tc, grp in df.groupby('traffic_category'):
    plt.scatter(grp['distance_km'], grp['historical_travel_time_minutes'],
                alpha=0.4, label=tc, color=traffic_colors[tc], s=20)
plt.xlabel('Distance (km)')
plt.ylabel('Travel Time (minutes)')
plt.title('Distance vs Travel Time by Traffic Level')
plt.legend(title='Traffic')
plt.tight_layout()
plt.show()

## 6. Feature Correlation Matrix (ML Feature Engineering)

In [ ]:
numeric_cols = [
    'distance_km', 'historical_travel_time_minutes', 'stop_count',
    'average_speed_kmph', 'route_efficiency_score', 'weekend_flag', 'visit_duration_minutes'
]
corr = df[numeric_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../data/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Location Features - Regional Analysis and Store Density

In [ ]:
region_stats = df.groupby('region').agg(
    trips=('driver_id', 'count'),
    avg_travel_time=('historical_travel_time_minutes', 'mean'),
    avg_efficiency=('route_efficiency_score', 'mean'),
    unique_locations=('location_id', 'nunique')
).sort_values('trips', ascending=False)

print('Regional Analysis:')
print(region_stats.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
region_stats['unique_locations'].plot(kind='bar', ax=ax, color='#8b5cf6')
ax.set_title('Store Density by Region (Unique Locations per Region)')
ax.set_xlabel('Region')
ax.set_ylabel('Unique Locations')
plt.tight_layout()
plt.show()

## 8. Model Target Distribution - Travel Time and Efficiency Score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['historical_travel_time_minutes'], bins=40, color='#3b82f6', edgecolor='white')
axes[0].set_title('Target 1: Travel Time Distribution')
axes[0].set_xlabel('Travel Time (minutes)')
axes[0].set_ylabel('Frequency')
axes[1].hist(df['route_efficiency_score'], bins=40, color='#22c55e', edgecolor='white')
axes[1].set_title('Target 2: Route Efficiency Score Distribution')
axes[1].set_xlabel('Efficiency Score (0-1)')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

print('Travel Time - Mean:', df['historical_travel_time_minutes'].mean().round(2), 'Std:', df['historical_travel_time_minutes'].std().round(2))
print('Efficiency Score - Mean:', df['route_efficiency_score'].mean().round(3), 'Std:', df['route_efficiency_score'].std().round(3))

## Summary
The dataset is well-suited for training the ensemble ML models:
- **Random Forest**: Handles tabular features with multi-output regression
- **XGBoost**: Provides high-accuracy single-target ETA prediction
- **LSTM**: Captures sequential stop patterns for next-stop prediction
- **K-Means**: Groups delivery zones geographically for fleet assignment